In [1]:
from selenium import webdriver
import time
import pandas
import requests
import numpy as np
import matplotlib.pyplot as plt
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
import time
from selenium.common.exceptions import MoveTargetOutOfBoundsException, StaleElementReferenceException
from selenium.webdriver.common.actions.wheel_input import ScrollOrigin
from selenium.webdriver.common.actions.wheel_input import ScrollOrigin
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By
import time
from bs4 import BeautifulSoup
from selenium import webdriver

In [2]:
options = webdriver.ChromeOptions()
from selenium import webdriver

options = webdriver.ChromeOptions()


options.add_argument("user-agent=ЗАЛУПА КРАКЕНА")
options.add_argument("--window-size=900,700")
options.add_argument("--disable-gpu")


options.add_argument("--disable-extensions")
options.add_argument("--disable-notifications")
options.add_argument("--disable-popup-blocking")


options.add_argument("--disable-background-timer-throttling")
options.add_argument("--disable-renderer-backgrounding")
options.add_argument("--disable-backgrounding-occluded-windows")
options.add_argument("--headless=new")

prefs = {
    "profile.managed_default_content_settings.images": 2
}
options.add_experimental_option("prefs", prefs)
# link = "https://www.cian.ru/sale/suburban/325680550/?mlSearchSessionGuid=6e63a0f2f58562fbca65613c8b233931"
link = "https://www.cian.ru/sale/suburban/315803826/?mlSearchSessionGuid=c001fef224acb60966e3109c22101ec9"
driver = webdriver.Chrome(options=options)



In [3]:
import pandas as pd

columns = [
    'link',         # ссылка
    'sellerType',   # тип продавца
    'posterName',   # название агенства/имя риэлтора
    'price',        # стоимость
    'houseSquare',  # площадь дома
    'areaSquare',   # площадь участка
    'address',      # адрес
    'distToKarp',   # расстояние до тебя
    'houseCondition'# состояние дома
    'photoAmount',  # кол-во фоток в объявлении
    'floorAmount',  # кол-во этажей
    'houseMaterial',# материал дома
    'areaStatus',   # статус участка (ИЖС/СНТ)
    'isSuperAgent', # является ли суперагентом
    'longitude',    # широта
    'latitude',     # долгота


]
global df
df = pd.DataFrame(columns=columns)

In [4]:
def get_price(soup):
    el = soup.select_one('[data-testid="price-amount"] span')
    if not el:
        return None

    return int(
        el.text
        .replace('\u202f', '')
        .replace('\xa0', '')
        .replace('₽', '')
        .replace(' ', '')
    )
import re

def get_houseSquare(soup):
    block = soup.select_one('[data-name="ObjectFactoids"]')
    if not block:
        return None

    items = block.select('[data-name="ObjectFactoidsItem"]')

    if not items:
        return None

    # первый item всегда площадь дома
    txt = items[0].get_text(" ", strip=True)

    m = re.search(r'([\d.,]+)\s*м', txt)
    if not m:
        return None

    return float(m.group(1).replace(',', '.'))

import re

import re

def get_areaSquare(soup):
    block = soup.select_one('[data-name="ObjectFactoids"]')
    if not block:
        return None

    # ищем среди карточек характеристик, а не по всей странице
    items = block.select('[data-name="ObjectFactoidsItem"]')
    if not items:
        return None

    for it in items:
        txt = it.get_text(" ", strip=True).lower()
        # участок обычно в "сот."
        m = re.search(r'([\d.,]+)\s*сот', txt)
        if m:
            try:
                return float(m.group(1).replace(',', '.'))
            except ValueError:
                return None

    return None

def get_houseMaterial(soup):
    block = soup.select_one('[data-name="ObjectFactoids"]')
    if not block:
        return None

    # ищем любой span, где есть текст "Материал дома"
    key = block.find(lambda tag: tag.name in ('span', 'div') and tag.get_text(strip=True) == 'Материал дома')
    if not key:
        return None

    item = key.find_parent(attrs={'data-name': 'ObjectFactoidsItem'})
    if not item:
        return None

    spans = item.find_all('span')
    if not spans:
        return None

    val = spans[-1].get_text(' ', strip=True)
    return val if val and val != 'Материал дома' else None

def get_floorAmount(soup):
    for span in soup.find_all('span'):
        text = span.get_text(strip=True)
        if text.isdigit():
            return int(text)
    return None

import re

def get_sellerType(soup):
    # 1) Пытаемся попасть именно в "карточку продавца" справа
    #    На собственнике это HomeownerLayout, на агентстве — AgencyBrandingAsideCardComponent
    seller_root = (
        soup.find(attrs={"data-name": "HomeownerLayout"}) or
        soup.find(attrs={"data-name": "AgencyBrandingAsideCardComponent"}) or
        soup.find(attrs={"data-name": "AuthorAside"}) or
        soup
    )

    txt = seller_root.get_text(" ", strip=True).lower()

    # 2) ТВОЯ ЛОГИКА ПРО СОБСТВЕННИКА — ОСТАВЛЯЕМ
    if "собственник" in txt:
        return "собственник"

    # 3) ДОБАВЛЯЕМ АГЕНТСТВО (как на скрине: "АГЕНТСТВО НЕДВИЖИМОСТИ")
    if ("агентство недвижимости" in txt) or ("агентство" in txt and "недвиж" in txt):
        return "агентство"

    # 4) (опционально) риелтор/риэлтор
    if re.search(r"\bри[её]лтор\b", txt):
        return "риелтор"

    return None

    t = b.get_text(" ", strip=True).lower()

    if 'собственник' in t:
        return 'Собственник'
    if 'агентство недвижимости' in t or 'агентство' in t:
        return 'Агенство'
    if 'риэлтор' in t:
        return 'Риэлтор'

    return None

def get_isSuperAgent(soup):
    block = soup.select_one(
        '[data-name="AuthorBrandingAside"], [data-testid="AgencyBrandingAsideCard"]'
    )
    if not block:
        return False

    return 'суперагент' in block.get_text(" ", strip=True).lower()

import re

def get_coords(html):
    m = re.search(r'"coordinates"\s*:\s*\{\s*"lat"\s*:\s*([0-9.]+)\s*,\s*"lng"\s*:\s*([0-9.]+)\s*\}', html)
    if not m:
        return None, None
    return float(m.group(1)), float(m.group(2))

def get_houseStatus(soup):
    groups = soup.find_all("div", {"data-name": "OfferSummaryInfoGroup"})
    for g in groups:
        items = g.find_all("div", {"data-name": "OfferSummaryInfoItem"})
        for it in items:
            ps = it.find_all("p")
            if len(ps) < 2:
                continue

            key = ps[0].get_text(strip=True)
            val = ps[1].get_text(strip=True)

            if key == "Состояние дома":
                return val

    return None

import re

def get_photoAmount(soup):
    # 1) самый стабильный вариант: ищем любой span, где есть "фото"
    for span in soup.find_all("span"):
        txt = span.get_text(" ", strip=True).lower()
        if "фото" not in txt:
            continue

        # ловим "10 фото" / "48 фото" / "10фото"
        m = re.search(r'(\d+)\s*фото', txt)
        if m:
            return int(m.group(1))

    # 2) запасной вариант: иногда бывает "фотографи(й/и): 10"
    text = soup.get_text(" ", strip=True).lower()
    m = re.search(r'фот[оа]\w*\s*[:\-]?\s*(\d+)', text)
    if m:
        return int(m.group(1))

    return None

def get_posterName(soup):
    # 1) агентство (как на скрине: есть ссылка и название агентства)
    agency = soup.select_one('[data-testid="AgencyBrandingAsideCard"] a[href] span')
    if agency:
        t = agency.get_text(" ", strip=True)
        if t:
            return t

    # 2) риэлтор (AgentInfoLayout: имя в a... class*="agent-name")
    realtor = soup.select_one('[data-name="AgentInfoLayout"] a[class*="agent-name"] span')
    if realtor:
        t = realtor.get_text(" ", strip=True)
        if t:
            return t

    # 3) собственник (HomeownerLayout: после "Собственник" обычно идёт ID/имя)
    home = soup.find(attrs={"data-name": "HomeownerLayout"})
    if home:
        spans = home.find_all("span")
        texts = [s.get_text(" ", strip=True) for s in spans if s.get_text(strip=True)]
        for i,txt in enumerate(texts):
            if txt.lower() == "собственник":
                # чаще всего следующий span — "ID ...."
                if i+1 < len(texts):
                    return texts[i+1]
                break

    return None

def get_address(soup):
    # 1) самый стабильный вариант: meta itemprop="name"
    m = soup.select_one('meta[itemprop="name"][content]')
    if m:
        t = m.get("content", "").strip()
        if t:
            return t

    # 2) fallback: собрать адрес из AddressItem ссылок
    parts = []
    for a in soup.select('address [data-name="AddressItem"]'):
        t = a.get_text(" ", strip=True)
        if t:
            parts.append(t)

    if parts:
        return ", ".join(parts)

    # 3) fallback: если вдруг всё иначе — берем первую строку address
    addr = soup.select_one("address")
    if addr:
        t = addr.get_text(" ", strip=True)
        return t if t else None

    return None

def get_areaStatus(soup):
    groups = soup.find_all("div", {"data-name": "OfferSummaryInfoGroup"})
    for g in groups:
        items = g.find_all("div", {"data-name": "OfferSummaryInfoItem"})
        for it in items:
            ps = it.find_all("p")
            if len(ps) < 2:
                continue

            key = ps[0].get_text(strip=True)
            val = ps[1].get_text(" ", strip=True)

            if key == "Статус участка":
                return val if val else None

    return None

In [5]:
# driver.get(url=link)
# time.sleep(1)
# soup = BeautifulSoup(driver.page_source, 'html.parser')
# price = get_price(soup)
# print(price)
# print(get_houseSquare(soup))
# print(get_areaSquare(soup))
# print(get_houseMaterial(soup))
# print(get_floorAmount(soup))
# print(get_sellerType(soup))
# print(get_isSuperAgent(soup))
# print(get_houseStatus(soup))
# html = driver.page_source
# latitude = get_coords(html)[0]
# longitude = get_coords(html)[1]
# print(latitude, longitude)
#
#


In [6]:
import math
import pandas as pd
from bs4 import BeautifulSoup

KARP_LAT = 55.777282
KARP_LNG = 37.586024

def dist_to_karp(lat, lng, karp_lat=KARP_LAT, karp_lng=KARP_LNG):
    if lat is None or lng is None:
        return None

    # haversine, км
    R = 6371.0
    p1 = math.radians(lat)
    p2 = math.radians(karp_lat)
    dp = math.radians(karp_lat - lat)
    dl = math.radians(karp_lng - lng)

    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

In [7]:
def read_links(path):
    with open(path, "r", encoding="utf-8") as f:
        links = [line.strip() for line in f if line.strip()]
    return links

links = read_links("links.txt")




In [8]:
import time
import random
from bs4 import BeautifulSoup

def get_offer_info(url):
    global df

    driver.get(url)
    time.sleep(random.uniform(3, 4))

    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")

    lat, lng = get_coords(html)

    # расстояние: если координаты не достались, чтобы не падало
    d = dist_to_karp(lat, lng) if (lat is not None and lng is not None) else None

    row = {
        "price": get_price(soup),
        "houseSquare": get_houseSquare(soup),
        "areaSquare": get_areaSquare(soup),
        "sellerType": get_sellerType(soup),
        "isSuperAgent": get_isSuperAgent(soup),
        "distToKarp": d,
        "link": url,
        "posterName": get_posterName(soup),
        "photoAmount": get_photoAmount(soup),
        "floorAmount": get_floorAmount(soup),
        "houseMaterial": get_houseMaterial(soup),
        "areaStatus": get_areaStatus(soup),
        "longitude": lng,
        "latitude": lat,
        "address": get_address(soup),
        "houseCondition": get_houseStatus(soup),
    }

    df.loc[len(df)] = row
    return row


# прогон по всем ссылкам (как у тебя сверху)
rows = []
for link in links:
    rows.append(get_offer_info(link))

df

,link,sellerType,posterName,price,houseSquare,areaSquare,address,distToKarp,houseConditionphotoAmount,floorAmount,houseMaterial,areaStatus,isSuperAgent,longitude,latitude
0,https://spb.cian.ru/sale/suburban/320726464/?c...,None,Егор Квач,19000000,195.0,8.0,"Санкт-Петербург, р-н Пушкинский, мкр. Пушкин, ...",616.169992,NaN,3,Газобетонный блок,Индивидуальное жилищное строительство,False,30.303534,59.736688
1,https://spb.cian.ru/sale/suburban/320726464/?c...,None,Егор Квач,19000000,195.0,8.0,"Санкт-Петербург, р-н Пушкинский, мкр. Пушкин, ...",616.169992,NaN,3,Газобетонный блок,Индивидуальное жилищное строительство,False,30.303534,59.736688
2,https://spb.cian.ru/sale/suburban/320726464/?c...,None,Егор Квач,19000000,195.0,8.0,"Санкт-Петербург, р-н Пушкинский, мкр. Пушкин, ...",616.169992,NaN,3,Газобетонный блок,Индивидуальное жилищное строительство,False,30.303534,59.736688
3,https://spb.cian.ru/sale/suburban/320726464/?c...,None,Егор Квач,19000000,195.0,8.0,"Санкт-Петербург, р-н Пушкинский, мкр. Пушкин, ...",616.169992,NaN,3,Газобетонный блок,Индивидуальное жилищное строительство,False,30.303534,59.736688
4,https://spb.cian.ru/sale/suburban/322054680/?c...,агентство,Агентство недвижимости Виктории Бердышевой,15100000,111.0,9.0,"Санкт-Петербург, р-н Приморский, Лахта-Ольгино...",645.380748,NaN,2,Каркасный,Садоводство,True,30.109679,60.017417
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1790,https://www.cian.ru/sale/suburban/327748719/?c...,агентство,Vira Group,18500000,90.0,6.7,"Москва, НАО (Новомосковский), Ветеран-2 СНТ, 89",32.736247,NaN,2,Газобетонный блок,Садоводство,True,37.492482,55.487652
1791,https://www.cian.ru/sale/suburban/327748888/?c...,агентство,Мой Дом,17645000,125.0,3.8,"Москва, НАО (Новомосковский), Макарово деревня",24.782204,NaN,2,Пенобетонный блок,Индивидуальное жилищное строительство,True,37.426896,55.573271
1792,https://www.cian.ru/sale/suburban/327748888/?c...,агентство,Мой Дом,17645000,125.0,3.8,"Москва, НАО (Новомосковский), Макарово деревня",24.782204,NaN,2,Пенобетонный блок,Индивидуальное жилищное строительство,True,37.426896,55.573271
1793,https://www.cian.ru/sale/suburban/327748916/?c...,агентство,ДОМЛАБ,45000000,350.0,9.0,"Москва, ТАО (Троицкий), № 452 квартал, 215",38.193304,NaN,2,Кирпичный,Садоводство,False,37.255461,55.488907


In [9]:
df.to_excel("cian_houses.xlsx", index=False, engine="openpyxl")